In [0]:
%pip install xgboost -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
import pandas as pd, numpy as np

S = "workspace.olist"
fact = spark.table(f"{S}.silver_order_fact").filter("order_status = 'delivered' AND is_late IS NOT NULL")
orders_m = (fact.groupBy("order_id").agg(
        F.count("*").alias("n_items"),
        F.sum("price").alias("items_price"),
        F.sum("freight_value").alias("freight_total"),
        F.sum("product_weight_g").alias("total_weight_g"),
        F.countDistinct("seller_id").alias("n_sellers"),
        F.max("estimated_days").alias("estimated_days"),
        F.max("max_installments").alias("max_installments"),
        F.min("customer_state").alias("customer_state"),
        F.min("seller_state").alias("seller_state"),
        F.min("main_payment_type").alias("main_payment_type"),
        F.min("order_purchase_timestamp").alias("purchase_ts"),
        F.max("is_late").alias("is_late"))
    .withColumn("same_state", (F.col("customer_state") == F.col("seller_state")).cast("int"))
    .withColumn("purchase_month", F.month("purchase_ts"))
    .withColumn("purchase_dow", F.dayofweek("purchase_ts"))
    .drop("purchase_ts"))

pdf = orders_m.toPandas()
pdf["total_weight_g"] = pdf["total_weight_g"].fillna(pdf["total_weight_g"].median())
pdf["max_installments"] = pdf["max_installments"].fillna(1)
pdf["main_payment_type"] = pdf["main_payment_type"].fillna("unknown")
X = pd.get_dummies(pdf.drop(columns=["order_id", "is_late"]),
                   columns=["customer_state", "seller_state", "main_payment_type"], dtype=int).astype(float)
y = pdf["is_late"].astype(int)
print(X.shape, "late rate:", round(y.mean(), 4))

from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

(96470, 64) late rate: 0.0811


In [0]:
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score, recall_score, precision_score
from xgboost import XGBClassifier

user = spark.sql("select current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/olist_late_delivery")

def evaluate(model, name, params):
    with mlflow.start_run(run_name=name):
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_te)[:, 1]
        pred = (proba >= 0.5).astype(int)
        m = {"roc_auc": roc_auc_score(y_te, proba), "pr_auc": average_precision_score(y_te, proba),
             "f1": f1_score(y_te, pred), "precision": precision_score(y_te, pred),
             "recall": recall_score(y_te, pred)}
        mlflow.log_params(params)
        mlflow.log_metrics(m)
        print(name, {k: round(v, 4) for k, v in m.items()})

lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
evaluate(lr, "logistic_regression_baseline", {"model": "logreg", "class_weight": "balanced"})

spw = round(float((y_tr == 0).sum() / (y_tr == 1).sum()), 2)
xgb_params = dict(n_estimators=300, max_depth=5, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw)
xgb = XGBClassifier(**xgb_params, eval_metric="auc", n_jobs=-1, random_state=42)
evaluate(xgb, "xgboost", {"model": "xgboost", **xgb_params})

2026/10/28 09:56:14 INFO mlflow.tracking.fluent: Experiment with name '/Users/pansarikrishna68@gmail.com/olist_late_delivery' does not exist. Creating a new experiment.


logistic_regression_baseline {'roc_auc': 0.6979, 'pr_auc': 0.1699, 'f1': 0.2292, 'precision': 0.14, 'recall': 0.6326}
xgboost {'roc_auc': 0.7811, 'pr_auc': 0.2829, 'f1': 0.3035, 'precision': 0.1974, 'recall': 0.6556}


In [0]:
imp = pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
print(imp.round(4))

customer_state_SP    0.0836
same_state           0.0501
purchase_month       0.0474
customer_state_MG    0.0446
customer_state_PR    0.0426
customer_state_RJ    0.0412
customer_state_BA    0.0401
estimated_days       0.0317
n_sellers            0.0310
customer_state_AL    0.0308
dtype: float32
